## Chris Dhong
## Group 3
## Group Assignment Task 2

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [7]:
# Start Spark
spark = SparkSession.builder.appName("chris-jupyter-notebook").getOrCreate()
sc = spark.sparkContext

In [8]:
import argparse

parser = argparse.ArgumentParser()
parser.add_argument(
    "--is-local",
    type=str,
    default="true",
    help="Whether running in local mode",
)

args, unknown = parser.parse_known_args()
is_local = args.is_local.lower() == "true"
if is_local:
    prefix = "../data/processed/merged"
else:
    prefix = "gs://msds-694-cohort-14-3/data"

print(f"Is local environment: {is_local}")

num_csv_path = f"{prefix}/num_2020.csv"
pre_csv_path = f"{prefix}/pre_2020.csv"
sub_csv_path = f"{prefix}/sub_2020.csv"
tag_csv_path = f"{prefix}/tag_2020.csv"

print(f"Num CSV path: {num_csv_path}")
print(f"Pre CSV path: {pre_csv_path}")
print(f"Sub CSV path: {sub_csv_path}")
print(f"Tag CSV path: {tag_csv_path}")

num_rdd = sc.textFile(num_csv_path)
pre_rdd = sc.textFile(pre_csv_path)
sub_rdd = sc.textFile(sub_csv_path)
tag_rdd = sc.textFile(tag_csv_path)

# print size of each RDD
print(f"Num RDD size: {num_rdd.count()}")
print(f"Pre RDD size: {pre_rdd.count()}")
print(f"Sub RDD size: {sub_rdd.count()}")
print(f"Tag RDD size: {tag_rdd.count()}")

Is local environment: True
Num CSV path: ../data/processed/merged/num_2020.csv
Pre CSV path: ../data/processed/merged/pre_2020.csv
Sub CSV path: ../data/processed/merged/sub_2020.csv
Tag CSV path: ../data/processed/merged/tag_2020.csv


Num RDD size: 11493263
Pre RDD size: 2746310
Sub RDD size: 24940
Tag RDD size: 298803


In [9]:
print("Spark started")

# Load chunk as DataFrame
df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(num_csv_path)
)

print("Loaded:", num_csv_path)
print("Row count:", df.count())
df.show(5, truncate=False)
df.printSchema()


Spark started


Loaded: ../data/processed/merged/num_2020.csv
Row count: 11493262
+--------------------+---------------------------------------------------+------------+--------+----+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------+---------+--------+-------+----+
|adsh                |tag                                                |version     |ddate   |qtrs|uom|segments                                                                                                                                                             |coreg                 |value    |footnote|quarter|year|
+--------------------+---------------------------------------------------+------------+--------+----+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------

In [10]:
# Column names for nums file (num_2020_chunk.csv)
TAG_COL = "tag"  # XBRL tag
COMPANY_COL = "adsh"  # filing ID (acts like entity/filing)
DATE_COL = "ddate"  # date in YYYYMMDD format

# Q1: Which tags appear most often in this subset?
q1 = (
    df.groupBy(TAG_COL)
    .agg(F.count("*").alias("num_rows"))
    .orderBy(F.desc("num_rows"))
)

print("=== Q1: rows by tag ===")
q1.show(20, truncate=False)

# Q2: How many unique filings, and which filings have the most rows?
q2_unique = df.select(COMPANY_COL).distinct().count()
print(f"\n=== Q2: unique filings (adsh) ===\nUnique IDs: {q2_unique}")

q2_top = (
    df.groupBy(COMPANY_COL)
    .agg(F.count("*").alias("num_rows"))
    .orderBy(F.desc("num_rows"))
)

print("\nTop filings by number of rows:")
q2_top.show(20, truncate=False)

# Q3: How many rows per year (using ddate)?
df_with_year = df.withColumn(
    "year", F.year(F.to_date(F.col(DATE_COL).cast("string"), "yyyyMMdd"))
)

q3 = (
    df_with_year.groupBy("year")
    .agg(F.count("*").alias("num_rows"))
    .orderBy("year")
)

print("\n=== Q3: rows per year ===")
q3.show(50, truncate=False)


=== Q1: rows by tag ===


+------------------------------------------------------------------------------------------------+--------+
|tag                                                                                             |num_rows|
+------------------------------------------------------------------------------------------------+--------+
|StockholdersEquity                                                                              |477807  |
|RevenueFromContractWithCustomerExcludingAssessedTax                                             |403939  |
|StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest                          |279204  |
|NetIncomeLoss                                                                                   |251066  |
|Revenues                                                                                        |197496  |
|ProfitLoss                                                                                      |138813  |
|OperatingIncomeLoss        


=== Q2: unique filings (adsh) ===
Unique IDs: 24939

Top filings by number of rows:


+--------------------+--------+
|adsh                |num_rows|
+--------------------+--------+
|0001193125-20-047353|8044    |
|0001558370-20-004766|5869    |
|0000004904-20-000084|4822    |
|0001109357-20-000053|4802    |
|0001109357-20-000207|4762    |
|0000004904-20-000062|4623    |
|0000930413-20-000859|4575    |
|0001109357-20-000165|4563    |
|0001664703-20-000013|4404    |
|0001213900-20-007917|4342    |
|0000913778-20-000003|4326    |
|0000092122-20-000017|4138    |
|0001193125-20-135806|4105    |
|0001610520-20-000041|3864    |
|0001370368-20-000076|3822    |
|0000891478-20-000017|3772    |
|0001193125-20-026877|3734    |
|0001104659-20-045603|3619    |
|0000930413-20-000534|3529    |
|0000913778-20-000008|3505    |
+--------------------+--------+
only showing top 20 rows

=== Q3: rows per year ===


+----+--------+
|year|num_rows|
+----+--------+
|1932|4       |
|1977|2       |
|1987|4       |
|1988|9       |
|1989|8       |
|1990|8       |
|1991|5       |
|1992|5       |
|1993|10      |
|1994|13      |
|1995|21      |
|1996|25      |
|1997|21      |
|1998|28      |
|1999|37      |
|2000|34      |
|2001|37      |
|2002|20      |
|2003|47      |
|2004|81      |
|2005|78      |
|2006|110     |
|2007|143     |
|2008|107     |
|2009|176     |
|2010|314     |
|2011|730     |
|2012|1126    |
|2013|1489    |
|2014|2204    |
|2015|4913    |
|2016|54888   |
|2017|634658  |
|2018|1798063 |
|2019|5285527 |
|2020|3707920 |
|2021|304     |
|2022|58      |
|2023|18      |
|2024|4       |
|2026|1       |
|2027|1       |
|2029|11      |
+----+--------+

